In [1]:
!pip install numpy
!pip install pandas


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import numpy as np
import pandas as pd

MONGO_URI = "mongodb://10.3.32.15:27017/"
MONGO_DB = "main"
WAYPOINTS_COLLECTION = "waypoints_v3"

In [6]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[WAYPOINTS_COLLECTION]

print(f"Connected to {MONGO_DB}.{WAYPOINTS_COLLECTION}")

Connected to main.waypoints_v3


In [ ]:
# Query and analyze one document to understand device structure
sample_doc = collection.find_one()
print("Sample document structure:")
import json
print(json.dumps(sample_doc, indent=2, default=str)[:2000])  # Print first 2000 chars
print("\n" + "="*50)
print("Available fields:")
for key in sample_doc.keys():
    print(f"  - {key}: {type(sample_doc[key]).__name__}")

In [7]:
# Read addresses from CSV
df = pd.read_csv('/Users/sheelpatel/Documents/doorstepai/Cleaned/DoorstepUtilities-Personal/AddressToBuildingIDs/addresses.csv')
print(f"Loaded {len(df)} addresses")
print(df.head())

Loaded 640 addresses
                              Address
0    713 E 211th St, Bronx, NY, 10467
1   1609 E 174th St, Bronx, NY, 10472
2  1330 Webster Ave, Bronx, NY, 10456
3      4439 3rd Ave, Bronx, NY, 10457
4    530 E 169th St, Bronx, NY, 10456


In [ ]:
# Lookup buildingIDs and device info - group by device
import re

# Connect to tracking_sessions_v2 for device info
tracking_collection = db["tracking_sessions_v2"]

results = []

for idx, row in df.iterrows():
    full_address = row['Address']
    # Extract base address (before first comma)
    base_address = full_address.split(',')[0].strip()
    
    # Use aggregation with regex to find matching documents (check main address and knownAddresses)
    pipeline = [
        {
            '$match': {
                '$or': [
                    {
                        'address': {
                            '$regex': re.escape(base_address),
                            '$options': 'i'  # Case insensitive
                        }
                    },
                    {
                        'knownAddresses.address': {
                            '$regex': re.escape(base_address),
                            '$options': 'i'  # Case insensitive
                        }
                    }
                ]
            }
        },
        {
            '$limit': 1
        }
    ]
    
    result = list(collection.aggregate(pipeline))
    
    if result:
        building_id = result[0].get('_id', 'NOT_FOUND')
        
        # Find device info for this building from tracking sessions
        devices_for_building = tracking_collection.find(
            {'buildingData.targetBuilding.buildingID': building_id},
            {'metadata.device': 1}
        )
        
        devices = []
        for session in devices_for_building:
            device = session.get('metadata', {}).get('device', 'Unknown')
            if device not in devices:
                devices.append(device)
        
        results.append({
            'Address': full_address,
            'BuildingID': building_id,
            'Devices': ', '.join(devices) if devices else 'No sessions found'
        })
        print(f"Found: {full_address} -> {building_id} | Devices: {len(devices)}")
    else:
        results.append({
            'Address': full_address,
            'BuildingID': 'NOT_FOUND',
            'Devices': 'N/A'
        })
        print(f"Not found: {full_address}")

results_df = pd.DataFrame(results)
print(f"\nProcessed {len(results_df)} addresses")
print(f"Found: {len(results_df[results_df['BuildingID'] != 'NOT_FOUND'])}")
print(f"Not found: {len(results_df[results_df['BuildingID'] == 'NOT_FOUND'])}")
print(results_df.head(10))

In [ ]:
# Save results to CSV with device info
output_path = '/Users/sheelpatel/Documents/doorstepai/Cleaned/DoorstepUtilities-Personal/AddressToBuildingIDs/addresses_with_buildingIDs.csv'
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")
print(f"Total addresses processed: {len(results_df)}")
print(f"Found: {len(results_df[results_df['BuildingID'] != 'NOT_FOUND'])}")
print(f"Not found: {len(results_df[results_df['BuildingID'] == 'NOT_FOUND'])}")